## Algorithms for Massive Dataset - project submission notebook

- Author: Andrea Colombo

### Project description

This notebook contains the implementation of the pagerank algorithm in python

### Step 1 - Environment setup

Here we simply pull in the required dependencies for the project, colab offers a wide variety of them but we need a couple extra ones to make this work
Some of these may be already included in the colab runtime, we pip install them just to be safe and avoid errors in the future

In [1]:
%pip install pyspark

In [2]:
import os

# This project needs a kaggle key in order to download the dataset, please set
# your own username and key in the lines below.
# NOTE: Do not commit the modified code anywhere public or your key will be
# leaked
os.environ['KAGGLE_USERNAME'] = 'x'
os.environ['KAGGLE_KEY'] = 'x'

if os.environ['KAGGLE_USERNAME'] == 'x' or os.environ['KAGGLE_KEY'] == 'x':
  print('================== WARNING ==========================')
  print('* Provide values for KAGGLE_KEY and KAGGLE_USERNAME *')
  print('================== WARNING ==========================')

================== WARNING ==========================
* Provide values for KAGGLE_KEY and KAGGLE_USERNAME *
================== WARNING ==========================


In [3]:
!kaggle datasets download -d benjaminawd/new-york-times-articles-comments-2020
!unzip -o new-york-times-articles-comments-2020.zip

Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
new-york-times-articles-comments-2020.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  new-york-times-articles-comments-2020.zip
  inflating: nyt-articles-2020.csv   
  inflating: nyt-comments-2020.csv   
  inflating: nyt-comments-part0.csv  
  inflating: nyt-comments-part1.csv  
  inflating: nyt-comments-part2.csv  
  inflating: nyt-comments-part3.csv  
  inflating: nyt-comments-part4.csv  
  inflating: nyt-comments-part5.csv  
  inflating: nyt-comments-part6.csv  
  inflating: nyt-comments-part7.csv  
  inflating: nyt-comments-part8.csv  
  inflating: nyt-comments-part9.csv  
  inflating: test.csv                
  inflating: train.csv               


In [4]:
# These variables are used to tweak some parts of the notebook execution, some
# parts (like sql self joins) are very time consuming when run with the entire
# dataset. If you do not want to wait for them, you can run the notebook with a
# fraction of the dataset
# Just enable the partial dataset with RUN_WITH_ENTIRE_DATASET and set the value
# of PARTIAL_DATASET_PROPORTION to a desired value (between 0.0 and 1.0)
RUN_WITH_ENTIRE_DATASET = False
PARTIAL_DATASET_PROPORTION = 0.2
RANDOM_SEED = 42

assert PARTIAL_DATASET_PROPORTION >= 0 and PARTIAL_DATASET_PROPORTION <= 1

if RUN_WITH_ENTIRE_DATASET == False:
  print(f'[INFO] Running the notebook with {PARTIAL_DATASET_PROPORTION * 100}%')

[INFO] Running the notebook with 20.0%


## Step 2 - Retrieve project dataset

The project dataset is open and retrievable from the web, we use

## Step 3 - Setup spark context and dataset retrieval



In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark import StorageLevel

spark = (SparkSession.builder.appName('AMD-PROJECT').getOrCreate())
sc = spark.sparkContext
sc.setLogLevel('WARN')

print(f'Loaded spark with version {spark.version}')

Loaded spark with version 4.0.4


In [6]:
# The initial data frames are computed using the basic csv parsing provided by
# pyspark, we are going to filter these later to prepare them for the pagerank
# implementation

articles_df = (spark
               .read
               .option('header', True)
               .option('inferSchema', True)
               .option('mode', 'PERMISSIVE')
               .csv('nyt-articles-2020.csv'))

comments_df = (spark
               .read
               .option('header', True)
               .option('inferSchema', True)
               .option('multiline', True)
               .option('quote', '"')
               .option('escape', '"')
               .csv('nyt-comments-2020.csv'))


print('===== Articles data frame snippet =====')
print('columns ', articles_df.columns)
# articles_df.show(10, truncate=True)

print('===== Comments data frame snippet =====')
print('columns ', comments_df.columns)
# comments_df.show(10, truncate=True)

===== Articles data frame snippet =====
columns  ['newsdesk', 'section', 'subsection', 'material', 'headline', 'abstract', 'keywords', 'word_count', 'pub_date', 'n_comments', 'uniqueID']
===== Comments data frame snippet =====
columns  ['commentID', 'status', 'commentSequence', 'userID', 'userDisplayName', 'userLocation', 'userTitle', 'commentBody', 'createDate', 'updateDate', 'approveDate', 'recommendations', 'replyCount', 'editorsSelection', 'parentID', 'parentUserDisplayName', 'depth', 'commentType', 'trusted', 'recommendedFlag', 'permID', 'isAnonymous', 'articleID']


In [7]:
# We can discard a lot of fields for this coputation, we only need the article
# id and title from the articles dataframe and we can discard the rest
# We also apply some basic duplicate cleaning and drop the null lines (although
# there shouldn't be any, we just do it to be sure we do not have runtime halts)

articles_cleaned = (articles_df
                    .select(
                        F.col('uniqueID').cast('string').alias('article_id'),
                        # The articles in this dataset do not have a proper
                        # title attribute so we use the headline instead
                        F.col('headline').cast('string').alias('article_title')
                    )
                    .filter(F.col('article_id').isNotNull())
                    # This check was added because some entries in the dataset
                    # seem to have 0 for ID (which doesn't really make sense)
                    # Snippet from tests:
                    # +--------------------+---------+
                    # |          article_id|  user_id|
                    # +--------------------+---------+
                    # |nyt://article/8bf...| 29778312|
                    # |nyt://article/8bf...| 10664125|
                    # |                   0| 65384034| <==== ????????????
                    # |nyt://article/176...| 75863404|
                    # |nyt://article/176...| 13800483|
                    # |nyt://article/176...| 59016607|
                    # |nyt://article/b31...| 69319909|
                    # |nyt://article/d56...| 79139306|
                    # |nyt://article/c95...| 58973206|
                    # |nyt://article/070...|102239845|
                    # +--------------------+---------+
                    #
                    # NOTE: with further testing I discovered that the article
                    # with the ID equal to 0 is the following (retrived using
                    # articles_cleaned.filter(F.col('article_id')=='0').show())
                    # +----------+--------------------+
                    # |article_id|       article_title|
                    # +----------+--------------------+
                    # |         0|Ben & Jerry’s Rad...|
                    # +----------+--------------------+
                    # Let's keep this check since this might be an issue with
                    # the dataset that may carry over with other versions
                    .filter(F.col('article_id') != '0')
                    .dropDuplicates(['article_id']))

comments_cleaned = (comments_df
                    .select(
                        F.col('articleID').cast('string').alias('article_id'),
                        F.col('userID').cast('string').alias('user_id')
                    )
                    .filter(F.col('user_id').isNotNull())
                    .filter(F.col('article_id').isNotNull())
                    .dropDuplicates(['article_id', 'user_id']))

if RUN_WITH_ENTIRE_DATASET == False:
  articles_cleaned = articles_cleaned.sample(
      fraction=PARTIAL_DATASET_PROPORTION,
      seed=RANDOM_SEED)
  comments_cleaned = comments_cleaned.sample(
      fraction=PARTIAL_DATASET_PROPORTION,
      seed=RANDOM_SEED)

articles_cleaned.printSchema()
comments_cleaned.printSchema()

root
 |-- article_id: string (nullable = true)
 |-- article_title: string (nullable = true)

root
 |-- article_id: string (nullable = true)
 |-- user_id: string (nullable = true)



## Step 4 - Construct the graph

In [8]:
joined_df = (comments_cleaned
             .join(
                 articles_cleaned.select('article_id'),
                 on='article_id',
                 how='inner'
             )
             .persist(StorageLevel.MEMORY_AND_DISK))

joined_df.printSchema()
joined_df.show(10, truncate=True)

# Now we construct the nodes of the graph we are going to operate on.
# In our graph, two articles (nodes) are connected by an edge if they
# share at least two users that commented.
# Two is a completely arbitrary threshold and it can be changed depending
# on the use case
#
# Graph -> G(V, E)
# given two arbitrary nodes i and j, with i != j, we have that:
# V(i,j) exists iff |CommentUserIDs(i) <intersect> CommentUserIDs(j)| >= 2
# V(i,j) exists => V(j,i) exists (undirected graph). The graph is undirected
# because there is no notion for the directionality of the graph edges (unlike
# for web pages, where a link has an intrinsic directionality)
graph_nodes_df = (joined_df
                  .select('article_id')
                  .distinct()
                  .persist(StorageLevel.MEMORY_AND_DISK))

print("Articles in the graph (nodes) = ", graph_nodes_df.count())
print("Unique Article-User pairs = ", joined_df.count())

root
 |-- article_id: string (nullable = true)
 |-- user_id: string (nullable = true)

+--------------------+---------+
|          article_id|  user_id|
+--------------------+---------+
|nyt://article/6e6...|100953244|
|nyt://article/118...| 63560021|
|nyt://article/118...| 27546635|
|nyt://article/7bb...| 87453573|
|nyt://article/375...| 20166476|
|nyt://article/574...| 73491210|
|nyt://article/0c1...| 64920544|
|nyt://article/afd...| 72369118|
|nyt://article/007...| 55823171|
|nyt://article/008...| 61085221|
+--------------------+---------+
only showing top 10 rows
Articles in the graph (nodes) =  3099
Unique Article-User pairs =  144633


In [9]:
# In order to construct the edges, we need to first get the articles that have
# been commented on by the same users and then apply the threshold of 2 to keep
# only the relevant ones
a = joined_df.alias('a')
b = joined_df.alias('b')

graph_edges_df = (a
                  .join(b, (
                      (F.col('a.user_id') == F.col('b.user_id')) &
                      # Since this join is already big, we use < instead of !=
                      # for article_ids to avoid having to prune one of the two
                      # directions later
                      (F.col('a.article_id') < F.col('b.article_id'))))
                  .select(F.col('a.article_id').alias('src_id'),
                          F.col('b.article_id').alias('dst_id'))
                  .groupBy('src_id', 'dst_id')
                  .agg(F.count('*').alias('shared_ids'))
                  .filter(F.col('shared_ids') >= 2))
                  # .persist(StorageLevel.MEMORY_AND_DISK)) -> This line has a
                  # bug in pyspark and leads to a null pointer exception :)
                  # TODO: check if it is a mistake or should be reported to the
                  # maintainers

graph_edges_df.show(10, truncate=True)

+--------------------+--------------------+----------+
|              src_id|              dst_id|shared_ids|
+--------------------+--------------------+----------+
|nyt://article/05a...|nyt://article/0bc...|         3|
|nyt://article/0b5...|nyt://article/565...|         4|
|nyt://article/693...|nyt://article/cf0...|        10|
|nyt://article/76c...|nyt://article/f64...|         3|
|nyt://article/ac1...|nyt://article/f45...|         3|
|nyt://article/d34...|nyt://interactive...|         2|
|nyt://article/08d...|nyt://article/ae6...|         4|
|nyt://article/0b5...|nyt://article/6a5...|         2|
|nyt://article/879...|nyt://article/e6d...|         2|
|nyt://article/8c7...|nyt://article/cd2...|         4|
+--------------------+--------------------+----------+
only showing top 10 rows
